In [46]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

# Statsmodels for interpretable logistic regression output
import statsmodels.api as sm

# Scikit-learn
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, log_loss, precision_recall_curve
)

In [23]:
df = pd.read_csv("../data/data_preprocees_rf.csv")
df

,age,tenure_months,annual_income_eur,credit_score,num_transactions_30d,avg_amount_30d_eur,max_amount_30d_eur,days_since_last_login,support_tickets_90d,chargebacks_12m,...,signup_year,signup_month,signup_day_of_week,days_since_signup,tx_per_month,tx_to_income_ratio,chargeback_rate,failed_payment_rate,composite_risk_score,target_is_fraud
0,34,82,38635.01,544.0,20,60.92,80.16,4.9,1,0,...,2025,11,4,101,0.228916,0.018916,0.000000,0.024390,0.338043,0
1,48,2,19912.97,703.0,21,112.11,571.12,0.3,0,0,...,2025,6,5,254,7.000000,0.067519,0.000000,0.000000,-2.474067,0
2,27,0,20326.87,720.0,25,73.61,492.57,4.6,1,0,...,2025,7,5,226,24.000000,0.043430,0.000000,0.000000,-1.356393,0
3,45,49,38452.47,703.0,17,47.53,204.18,25.3,1,0,...,2024,7,1,566,0.340000,0.014828,0.000000,0.028571,0.338043,0
4,37,46,31943.14,594.0,13,99.95,734.09,12.8,0,1,...,2025,5,2,271,0.276596,0.037534,0.013699,0.000000,2.005874,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
159995,56,0,34775.62,727.0,21,51.72,226.11,3.8,0,0,...,2025,7,2,222,21.000000,0.017841,0.000000,0.000000,-2.474067,0
159996,41,4,88617.57,770.0,18,52.54,171.07,15.1,1,0,...,2024,7,5,576,3.600000,0.007114,0.000000,0.000000,1.239936,0
159997,30,2,41148.54,738.0,20,29.34,119.81,0.7,0,0,...,2024,8,4,549,6.666667,0.008554,0.000000,0.000000,-2.474067,0
159998,56,6,31943.14,719.0,25,88.56,553.16,22.6,2,0,...,2025,3,2,341,3.571429,0.033257,0.000000,0.000000,-0.238719,0


In [24]:
from statsmodels.stats import descriptivestats

df.describe()

,age,tenure_months,annual_income_eur,credit_score,num_transactions_30d,avg_amount_30d_eur,max_amount_30d_eur,days_since_last_login,support_tickets_90d,chargebacks_12m,...,signup_year,signup_month,signup_day_of_week,days_since_signup,tx_per_month,tx_to_income_ratio,chargeback_rate,failed_payment_rate,composite_risk_score,target_is_fraud
count,160000.000000,160000.000000,160000.000000,160000.000000,160000.000000,160000.000000,160000.000000,160000.000000,160000.000000,160000.000000,...,160000.000000,160000.000000,160000.000000,160000.000000,160000.000000,160000.000000,160000.000000,160000.000000,160000.000000,160000.000000
mean,38.180700,17.747238,36660.592187,670.094206,22.010288,60.783112,292.509249,11.898687,0.786775,0.048763,...,2024.498575,6.494819,2.989081,413.047650,3.123338,0.025967,0.000555,0.007991,-0.013781,0.030775
std,11.440212,17.020847,20308.522522,57.395624,4.622836,39.147629,237.971011,11.452111,0.858241,0.215372,...,0.500000,3.440732,2.001009,211.038192,4.124815,0.022956,0.002484,0.013583,2.186616,0.172708
min,18.000000,0.000000,8840.598500,531.000000,12.000000,5.460000,17.959800,0.100000,0.000000,0.000000,...,2024.000000,1.000000,0.000000,55.000000,0.228916,0.001470,0.000000,0.000000,-2.474067,0.000000
25%,30.000000,5.000000,22769.945000,632.000000,19.000000,32.320000,122.220000,3.500000,0.000000,0.000000,...,2024.000000,4.000000,1.000000,230.000000,0.818182,0.010429,0.000000,0.000000,-1.356393,0.000000
50%,38.000000,12.000000,31943.140000,670.000000,22.000000,52.540000,224.730000,8.300000,1.000000,0.000000,...,2024.000000,7.000000,3.000000,413.000000,1.600000,0.019268,0.000000,0.000000,-0.238719,0.000000
75%,46.000000,25.000000,44988.015000,708.000000,25.000000,79.820000,389.300000,16.700000,1.000000,0.000000,...,2025.000000,9.000000,5.000000,597.000000,3.500000,0.033548,0.000000,0.018868,1.239936,0.000000
max,66.000000,82.000000,115836.969200,809.000000,34.000000,195.270200,1188.503000,55.400000,3.000000,1.000000,...,2025.000000,12.000000,6.000000,770.000000,24.000000,0.124763,0.013699,0.057143,6.799901,1.000000


In [25]:
import pandas as pd

df_numeric = df.select_dtypes(exclude=['object', 'category'])

df_numeric = df_numeric.dropna(axis=1)

print("Colonnes restantes :", df_numeric.columns.tolist())



Colonnes restantes : ['age', 'tenure_months', 'annual_income_eur', 'credit_score', 'num_transactions_30d', 'avg_amount_30d_eur', 'max_amount_30d_eur', 'days_since_last_login', 'support_tickets_90d', 'chargebacks_12m', 'failed_payments_6m', 'device_trust_z', 'ip_risk_z', 'is_vpn', 'num_devices_30d', 'is_new_device', 'channel', 'signup_source', 'plan_type', 'payment_method', 'browser', 'os', 'occupation', 'device_type', 'merchant_category', 'country', 'region', 'income_log', 'income_estimate_alt_eur', 'credit_score_norm', 'tx_amount_total_30d_eur', 'max_to_avg_ratio', 'internal_signal_1', 'internal_signal_2', 'internal_signal_3', 'internal_signal_4', 'internal_signal_5', 'internal_signal_6', 'internal_signal_7', 'internal_signal_8', 'terms_accepted_flag', 'partner_risk_indicator', 'legacy_partner_score', 'signup_year', 'signup_month', 'signup_day_of_week', 'days_since_signup', 'tx_per_month', 'tx_to_income_ratio', 'chargeback_rate', 'failed_payment_rate', 'composite_risk_score', 'target_

In [39]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler

for scaler in [StandardScaler(), RobustScaler(), MinMaxScaler()]:
    pipe = Pipeline([
        ("scaler", scaler),
        ("knn", KNeighborsClassifier(n_neighbors=7))
    ])

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    print(f"\n=== SCALER: {scaler.__class__.__name__} ===")
    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred))



=== SCALER: StandardScaler ===
[[38767     2]
 [ 1231     0]]
              precision    recall  f1-score   support

           0       0.97      1.00      0.98     38769
           1       0.00      0.00      0.00      1231

    accuracy                           0.97     40000
   macro avg       0.48      0.50      0.49     40000
weighted avg       0.94      0.97      0.95     40000


=== SCALER: RobustScaler ===
[[38767     2]
 [ 1230     1]]
              precision    recall  f1-score   support

           0       0.97      1.00      0.98     38769
           1       0.33      0.00      0.00      1231

    accuracy                           0.97     40000
   macro avg       0.65      0.50      0.49     40000
weighted avg       0.95      0.97      0.95     40000


=== SCALER: MinMaxScaler ===
[[38765     4]
 [ 1231     0]]
              precision    recall  f1-score   support

           0       0.97      1.00      0.98     38769
           1       0.00      0.00      0.00      123

In [41]:


pipe_smote = Pipeline([
    ("scaler", RobustScaler()),
    ("smote", SMOTE(random_state=42)),
    ("knn", KNeighborsClassifier(n_neighbors=7))
])

pipe_smote.fit(X_train, y_train)
y_pred_smote = pipe_smote.predict(X_test)

print("\n=== ROBUST + SMOTE ===")
print(confusion_matrix(y_test, y_pred_smote))
print(classification_report(y_test, y_pred_smote))



=== ROBUST + SMOTE ===
[[21453 17316]
 [  584   647]]
              precision    recall  f1-score   support

           0       0.97      0.55      0.71     38769
           1       0.04      0.53      0.07      1231

    accuracy                           0.55     40000
   macro avg       0.50      0.54      0.39     40000
weighted avg       0.94      0.55      0.69     40000



In [52]:
k_values = [3, 5, 7, 9, 11, 15, 21, 31, 35, 41, 45, 50, 100, 200]
results = []

for k in k_values:
    pipe = Pipeline([
        ("scaler", RobustScaler()),
        ("smote", SMOTE(random_state=42)),
        ("knn", KNeighborsClassifier(
            n_neighbors=k,
            weights="distance",
            metric="minkowski",
            p=2
        ))
    ])

    pipe.fit(X_train, y_train)

    y_proba = pipe.predict_proba(X_test)[:, 1]

    precision, recall, thresholds = precision_recall_curve(y_test, y_proba)
    f1_scores = 2 * precision * recall / (precision + recall + 1e-9)

    best_idx = np.argmax(f1_scores)
    best_f1 = f1_scores[best_idx]
    best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 1.0

    results.append({
        "k": k,
        "best_f1": best_f1,
        "best_threshold": best_threshold
    })

    print(
        f"k={k:<3} | F1_opt={best_f1:.4f} | seuil={best_threshold:.3f}"
    )


k=3   | F1_opt=0.0690 | seuil=0.319
k=5   | F1_opt=0.0692 | seuil=0.397
k=7   | F1_opt=0.0695 | seuil=0.567
k=9   | F1_opt=0.0694 | seuil=0.541
k=11  | F1_opt=0.0697 | seuil=0.722
k=15  | F1_opt=0.0707 | seuil=0.732
k=21  | F1_opt=0.0708 | seuil=0.662
k=31  | F1_opt=0.0714 | seuil=0.643
k=35  | F1_opt=0.0726 | seuil=0.736
k=41  | F1_opt=0.0727 | seuil=0.757
k=45  | F1_opt=0.0739 | seuil=0.764
k=50  | F1_opt=0.0740 | seuil=0.852
k=100 | F1_opt=0.0786 | seuil=0.841
k=200 | F1_opt=0.0875 | seuil=0.889


In [61]:
X = df_numeric.drop(columns="target_is_fraud")
y = df_numeric["target_is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ('classifier', RandomForestClassifier(n_estimators=200,class_weight='balanced',random_state=42))
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy : 0.96965
Confusion Matrix:
 [[38769     0]
 [ 1214    17]]

Classification Report:
               precision    recall  f1-score   support

           0       0.97      1.00      0.98     38769
           1       1.00      0.01      0.03      1231

    accuracy                           0.97     40000
   macro avg       0.98      0.51      0.51     40000
weighted avg       0.97      0.97      0.96     40000



In [ ]:
df_test = pd.read_csv("../data/data_preprocess_rf_test.csv")
df_test

,Unnamed: 0,age,tenure_months,annual_income_eur,credit_score,num_transactions_30d,avg_amount_30d_eur,max_amount_30d_eur,days_since_last_login,support_tickets_90d,...,device_type_te,merchant_category_te,country_te,region_te,city_te,last_ticket_subject_te,referrer_code_te,signup_date_te,manual_review_result_te,customer_id
0,15273,-0.018856,-0.389414,1.300338,-0.835725,0.139466,-0.732395,-0.199447,-0.824020,1.341507,...,0.030998,0.032750,0.031353,0.030207,0.035771,0.033182,0.03075,0.029146,0.029707,CUST_925MSOFELP
1,133842,0.491365,1.073572,-0.201193,-0.835725,0.028509,-1.175753,-0.758894,-0.882317,1.341507,...,0.030998,0.031283,0.031353,0.032106,0.031327,0.031046,0.03075,0.029146,0.551891,CUST_PTGSLVG75B
2,44927,0.151217,1.298647,-0.562247,0.667715,0.083988,-0.240240,-0.281545,-0.890645,0.224806,...,0.030998,0.028930,0.031353,0.029891,0.023841,0.033182,0.03075,0.029146,0.001797,CUST_S6ERZCDY77
3,83817,0.151217,0.060736,0.396121,-0.972401,-0.221143,-1.044643,-0.637294,-0.632472,0.224806,...,0.030998,0.028930,0.031353,0.028478,0.024327,0.022973,0.03075,0.029146,0.029707,CUST_7XFAN8C8LI
4,107431,-0.018856,0.060736,-0.201193,-0.084005,-0.054708,0.432560,-0.257989,1.041486,-0.891895,...,0.030998,0.032750,0.031353,0.033846,0.033493,0.031046,0.03075,0.029146,0.029707,CUST_8AO6C6Q06I
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31995,79938,0.406328,1.973871,-0.527976,1.966140,-0.193404,-0.849457,-0.997579,1.782692,0.224806,...,0.030695,0.032837,0.029796,0.029220,0.029632,0.028625,0.03075,0.030750,0.001797,CUST_K100QZMNN2
31996,55835,0.236254,-0.501951,-0.905006,0.189348,-0.110186,-1.457688,-0.281545,-0.099470,-0.891895,...,0.030428,0.029576,0.029796,0.029220,0.025303,0.033101,0.03075,0.030750,0.001797,CUST_QMHTARFSQP
31997,42846,0.066181,0.117005,-0.876653,1.128997,0.167205,0.048596,-0.281545,-0.749066,-0.891895,...,0.030998,0.032837,0.031353,0.032298,0.035853,0.031046,0.03075,0.030750,0.001797,CUST_S3QNAR2VT9
31998,96511,0.406328,-0.952100,2.057313,-0.630710,-0.221143,0.937777,0.594010,-0.532534,0.224806,...,0.030998,0.028930,0.031353,0.031510,0.036077,0.022973,0.03075,0.030750,0.001797,CUST_IZO6DIZHT6


In [35]:
y_test_pred = pipeline.predict(df_test) 


ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- Unnamed: 0
- browser_te
- channel_te
- chargeback_resolution_time_days
- city_te
- ...
Feature names seen at fit time, yet now missing:
- browser
- channel
- chargeback_rate
- composite_risk_score
- country
- ...


In [20]:
df_results = pd.DataFrame({
    "customer_id": df_test["customer_id"],  
    "target_is_fraud": y_test_pred          
})

print(df_results.head())

df_results.to_csv("../Preds/KNN_preds_v2_Noah.csv", index=False)

       customer_id  target_is_fraud
0  CUST_925MSOFELP                0
1  CUST_PTGSLVG75B                0
2  CUST_S6ERZCDY77                0
3  CUST_7XFAN8C8LI                1
4  CUST_8AO6C6Q06I                1


In [21]:
df_results.value_counts(ascending=False)

customer_id      target_is_fraud
CUST_RLT8DRGSYO  0                  2
CUST_ZKKTPKNVK4  0                  2
CUST_5LCUF5P1HQ  0                  2
CUST_4AWC0QTYNG  0                  2
CUST_2V9TV3O0Z6  0                  2
                                   ..
CUST_C4DDL6VZ51  0                  1
CUST_C4DAZ73QL7  0                  1
CUST_C4BZ6SK53X  0                  1
CUST_C4BXE4V2P4  0                  1
CUST_ZZY9LAHVBF  0                  1
Name: count, Length: 31602, dtype: int64